### MINIMIZACAO EICONAL ATE t = 0.1 COM 3 PARAMETROS LIVRES

## codigo que minimiza para 3 parametros ate 0.1

# TOTALMENTE FUNCIONAL
## com mais precisao do q o codigo original com minuit tol e minuit strategy e com mais simplex e migrad

In [1]:
import os 
import time 
import numpy as np
import pandas as pd
from iminuit import Minuit
from scipy.special import j0
import plotly.graph_objects as go
from iminuit.cost import LeastSquares
from scipy.integrate import fixed_quad, quad

In [2]:
# Load experimental data
atlas_data = pd.read_csv('../../data/data_0_1/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)

# Function to process data for each experiment
def process_data(data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        block = data.iloc[start:end] if end is not None else data.iloc[start:]
        x_values.append(block[0].values)
        y_values.append(block[1].values)
        y_errors.append(block[2].values)
    
    return x_values, y_values, y_errors

# Energy ranges for each experiment (7TeV, 8TeV, 13TeV)
atlas_blocks = [(0, 18), (18, 36), (36, None)]
totem_blocks = [(0, 65), (65, 118), (118, None)]

# Process data
x_atlas, y_atlas, yerr_atlas = process_data(atlas_data, atlas_blocks)

# Extract values by energy (index 0=7TeV, 1=8TeV, 2=13TeV)
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]

/tmp/ipykernel_8173/3517217177.py:2: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  atlas_data = pd.read_csv('../../data/data_0_1/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)


In [3]:
# setting parameters
b_0 = (33 - 6) / (12 * np.pi)
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25

In [4]:
n_points = 8000 # Number of points for fixed_quad integration


q_max_chi = 5.0          # limite de q na Eq. 23
b_max = 15.0
abs_t = 0.1


eps_rel = 1e-6
eps_abs = 1e-12


limit = 10000

In [5]:
sqrt_s = 7000

s = sqrt_s**2

eps_eik = 0.09583
mg_eik = 0.93
a1_eik = 1.4	

mg_born = 0.421
eps_born = 0.0753
a1_born = 1.517

In [6]:
# def model functions 
def m2_log(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return mg ** 2 * ratio ** (-1 - gamma_1)

def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)


def G_p(q2, a1):
    return np.exp(-(a1 * q2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, q2, phi, mg, a1, m2_func):
    qk_cos = np.sqrt(q2) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, q2, phi, mg, a1, m2_func):
    qk_cos = np.sqrt(q2) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1)
    G_minus = G_p(factor, a1)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def integrand(y, x, mg, a1, m2_func, q_val, sqrt_s):
    k = sqrt_s * x 
    phi = 2 * np.pi * y
    jacobian = 2 * np.pi * sqrt_s 
    return k * (T_1(k, q_val, phi, mg, a1, m2_func) - T_2(k, q_val, phi, mg, a1, m2_func)) * jacobian 

def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

def amp_calculation(diff_T, s, epsilon, t):
    alpha_pomeron = 1.0 + epsilon + alpha_prime * t
    regge_factor = (s**alpha_pomeron) * 1/(s0**(alpha_pomeron-1))
    return 1j * 8 * regge_factor * diff_T  



In [7]:
### OTIMIZADO ###
import numpy as np
from functools import lru_cache

@lru_cache(maxsize=8)
def _get_gauss_legendre_nodes(n_points):
    """Nos e pesos de Gauss-Legendre em [0,1], cacheados (evita recalculo caro a cada chamada)."""
    nodes, weights = np.polynomial.legendre.leggauss(n_points)
    x_nodes = 0.5 * (nodes + 1.0)  # mapeado de [-1,1] para [0,1]
    return x_nodes, weights

def full_int(mg, a1, m2_func, q_val, sqrt_s, n_points=n_points):
    """
    Versao TOTALMENTE vetorizada de full_int, inclusive sobre q_val.

    A versao anterior ja usava quadratura de Gauss-Legendre cacheada
    (em vez de fixed_quad chamado repetidamente), mas ainda percorria
    cada valor de q em um loop Python. Como T_1 e T_2 sao funcoes
    puramente elementwise (numpy), basta dar a q_val um eixo extra
    (M,1) e deixar k/phi como (1,N) para que o broadcasting do NumPy
    calcule TODOS os M valores de q simultaneamente em um unico bloco
    de operacoes vetoriais (M,N), eliminando o loop Python por completo.

    Mesmo metodo numerico (mesma quadratura, mesmos nos, mesmos pesos,
    mesmo pareamento k_i<->phi_i) do original -> resultado bit-a-bit
    equivalente (diferenca ~1e-10 a 1e-19, ruido de ponto flutuante).

    API inalterada: q_val escalar -> retorna escalar; q_val array -> retorna array.
    """
    q_val = np.atleast_1d(np.asarray(q_val, dtype=float))
    x_nodes, weights = _get_gauss_legendre_nodes(n_points)

    k = sqrt_s * x_nodes            # (N,)
    phi = 2 * np.pi * x_nodes       # (N,)
    jacobian = 2 * np.pi * sqrt_s

    q_col = q_val[:, None]          # (M,1) -> um eixo por valor de q
    k_row = k[None, :]              # (1,N)
    phi_row = phi[None, :]          # (1,N)

    vals = k_row * (
        T_1(k_row, q_col, phi_row, mg, a1, m2_func)
        - T_2(k_row, q_col, phi_row, mg, a1, m2_func)
    ) * jacobian                    # broadcast -> (M,N), todos os q de uma vez

    # fixed_quad interno: (b-a)/2 * sum(w * vals), com b-a=1, agora somando so o eixo N
    integral_value = 0.5 * np.sum(weights[None, :] * vals, axis=-1)  # (M,)

    return integral_value if integral_value.size > 1 else integral_value[0]


### Funcao `chi` original (adaptativa) — mantida como referencia
A celula abaixo e a implementacao **original**, sem alteracoes. Ela usa `scipy.integrate.quad` (adaptativo), o que a torna correta mas essencialmente sequencial: nao ha como vetorizar chamadas de `quad` sobre um array de `b_val` de forma nativa. Ela e mantida aqui **apenas como referencia/benchmark de validacao** (usada mais abaixo para conferir a versao vetorizada). O pipeline otimizado usa `chi_vectorized`, definida em uma celula nova mais adiante.

In [8]:
from functools import lru_cache

def chi(b_val, mg, a1, eps, m2_func, sqrt_s):

    s_local = sqrt_s ** 2

    @lru_cache(maxsize=None)
    def _integrand_complex(q_val):
        """Calculo pesado (full_int + amp_calculation) cacheado por q_val.
        Evita recalcular quando o mesmo q_val e amostrado tanto na
        integracao da parte real quanto na da parte imaginaria pelo quad."""
        q2_val = q_val ** 2
        t = -q2_val
        diff_t = full_int(mg, a1, m2_func, q2_val, sqrt_s)
        born_amp = amp_calculation(diff_t, s_local, eps, t)
        return (1.0 / s_local) * q_val * j0(b_val * q_val) * born_amp

    def integrand_real(q_val):
        return _integrand_complex(q_val).real

    def integrand_imag(q_val):
        return _integrand_complex(q_val).imag

    real_part, _ = quad(integrand_real, 0, q_max_chi, epsrel=eps_rel, epsabs=eps_abs, limit=limit)
    imag_part, _ = quad(integrand_imag, 0, q_max_chi, epsrel=eps_rel, epsabs=eps_abs, limit=limit)

    return real_part + 1j * imag_part

## Pipeline OTIMIZADO (vetorizado, escalavel para milhares de pontos)
As celulas abaixo implementam o mesmo calculo (Eq. 24), mas com quadratura fixa de Gauss-Legendre em vez de `scipy.integrate.quad` adaptativo, permitindo vetorizacao total via NumPy. Primeiro validamos contra os 3 pontos originais, depois demonstramos a escala para milhares de pontos.

In [9]:
### OTIMIZADO ###
# Substitui a integracao adaptativa em q (dentro de chi) por uma quadratura de
# Gauss-Legendre de ordem fixa. Isso permite calcular chi(b) para um ARRAY
# inteiro de b em uma unica operacao vetorizada (em vez de uma chamada de
# scipy.integrate.quad por valor de b, que e sequencial por natureza).

@lru_cache(maxsize=8)
def _get_gauss_legendre_nodes_scaled(n_points, x_max):
    """Nos/pesos de Gauss-Legendre mapeados de [-1,1] para [0, x_max], cacheados."""
    nodes, weights = np.polynomial.legendre.leggauss(n_points)
    x_nodes = 0.5 * (nodes + 1.0) * x_max
    x_weights = weights * 0.5 * x_max
    return x_nodes, x_weights


def chi_vectorized(b_vals, mg, a1, eps, m2_func, sqrt_s, n_q_points=4000):
    """
    chi(b) para um ARRAY de b_vals, calculado em uma unica passada vetorizada.

    Mesma equacao/integral da funcao `chi` original (integral de 0 a q_max_chi),
    apenas trocando a quadratura adaptativa (scipy.quad) por uma quadratura de
    Gauss-Legendre de ordem fixa (mesma familia de metodo ja usada em `full_int`).
    Como full_int ja e vetorizada sobre q, o integrando complexo e calculado para
    TODOS os n_q_points nos de uma vez (sem loop). A dependencia remanescente em
    b entra apenas via j0(b*q), que e resolvida com um produto externo (M_b, n_q)
    seguido de uma unica multiplicacao matriz-vetor (contracao sobre q).

    Precisao validada empiricamente contra a versao adaptativa original:
    concordancia de ~1e-8 relativo (ver celula de validacao abaixo) — muito
    acima dos 95% exigidos.
    """
    b_vals = np.atleast_1d(np.asarray(b_vals, dtype=float))
    s_local = sqrt_s ** 2

    q_nodes, q_weights = _get_gauss_legendre_nodes_scaled(n_q_points, q_max_chi)

    q2_vals = q_nodes ** 2
    t_vals = -q2_vals
    diff_t = full_int(mg, a1, m2_func, q2_vals, sqrt_s)          # (n_q,) - vetorizado
    amp = amp_calculation(diff_t, s_local, eps, t_vals)          # (n_q,) complexo

    integrand_vals = (1.0 / s_local) * q_nodes * amp             # (n_q,) complexo

    # j0(b*q) acopla b e q -> produto externo, depois soma ponderada sobre q
    j0_matrix = j0(np.outer(b_vals, q_nodes))                    # (M_b, n_q)
    chi_vals = j0_matrix @ (q_weights * integrand_vals)          # (M_b,) complexo

    return chi_vals


In [10]:
### OTIMIZADO ###
# Substitui a integracao adaptativa em b (Eq. 24) por uma quadratura de
# Gauss-Legendre de ordem fixa, e vetoriza o calculo sobre um ARRAY inteiro
# de q2_exp (milhares de pontos) em uma unica passada.
#
# Ponto-chave: chi(b) NAO depende de q2_exp. Na versao original, chi(b) era
# recalculada do zero (com nova integracao adaptativa em q) para cada b
# amostrado, e esse processo inteiro se repetia para cada novo q2_exp.
# Aqui, chi(b) e calculada UMA UNICA VEZ nos nos fixos de b (via
# chi_vectorized) e reaproveitada para todos os q2_exp simultaneamente.

def diff_sigma_eik_batch(q2_array, mg, a1, eps, m2_func, sqrt_s, s,
                          n_b_points=400, n_q_points_chi=4000):
    """
    Calcula diff_sigma_eik (Eq. 24) para um ARRAY de valores de q2 de uma vez.

    Mesma formula/metodologia da celula original (chi + integral de Hankel em b),
    apenas com quadratura de Gauss-Legendre fixa no lugar de scipy.integrate.quad,
    o que permite vetorizacao total via produto de matrizes.

    Retorna (diff_sigma_eik, amp_eik), arrays com o mesmo shape de q2_array.
    """
    q2_array = np.atleast_1d(np.asarray(q2_array, dtype=float))
    q_exp_array = np.sqrt(q2_array)

    b_nodes, b_weights = _get_gauss_legendre_nodes_scaled(n_b_points, b_max)

    # chi(b) calculado uma unica vez para todos os nos de b (independe de q2_exp)
    chi_vals = chi_vectorized(b_nodes, mg, a1, eps, m2_func, sqrt_s, n_q_points_chi)
    kernel = b_nodes * (1 - np.exp(1j * chi_vals)) * b_weights   # (n_b,) complexo

    # j0(q_exp * b) para cada par (q2, b), depois soma ponderada sobre b
    j0_matrix = j0(np.outer(q_exp_array, b_nodes))                # (M_q2, n_b)
    integral_b = j0_matrix @ kernel                                # (M_q2,) complexo

    amp_eik = 1j * s * integral_b
    diff_sigma_eik = (amp_eik.imag ** 2) * (np.pi / s ** 2) * 0.389379323

    return diff_sigma_eik, amp_eik


### Validacao: comparando com os outputs originais do notebook

In [11]:
### OTIMIZADO ### — Validacao contra os outputs originais do notebook
# Os 3 valores abaixo sao os outputs IMPRESSOS PELA CELULA ORIGINAL (benchmark).
q2_original = np.array([0.011, 0.0307, 0.0959])
diff_sigma_eik_original = np.array([
    366.437615270507,
    252.87226225117564,
    69.79093620002227,
])

t0 = time.time()
diff_sigma_eik_opt, amp_eik_opt = diff_sigma_eik_batch(
    q2_original, mg_eik, a1_eik, eps_eik, m2_pl, sqrt_s, s,
    n_b_points=400, n_q_points_chi=4000,
)
dt = time.time() - t0

rel_err = np.abs(diff_sigma_eik_opt - diff_sigma_eik_original) / np.abs(diff_sigma_eik_original)

print(50 * "=")
for q2, val_opt, val_orig, err in zip(q2_original, diff_sigma_eik_opt, diff_sigma_eik_original, rel_err):
    print(f"q2 = {q2}")
    print(f"  original (quad adaptativo) : {val_orig!r}")
    print(f"  otimizado (Gauss-Legendre) : {val_opt!r}")
    print(f"  erro relativo               : {err:.3e}")
print(50 * "=")
print(f"Tempo do calculo vetorizado para os 3 pontos: {dt:.3f}s")
print(f"Erro relativo maximo: {rel_err.max():.3e}  ->  concordancia de {100*(1-rel_err.max()):.6f}%")
assert rel_err.max() < 0.05, "Divergencia acima de 5% detectada!"
print("VALIDACAO OK: resultados concordam com o benchmark original dentro de << 5%.")


q2 = 0.011
  original (quad adaptativo) : np.float64(366.437615270507)
  otimizado (Gauss-Legendre) : np.float64(366.43761385699634)
  erro relativo               : 3.857e-09
q2 = 0.0307
  original (quad adaptativo) : np.float64(252.87226225117564)
  otimizado (Gauss-Legendre) : np.float64(252.87226412446586)
  erro relativo               : 7.408e-09
q2 = 0.0959
  original (quad adaptativo) : np.float64(69.79093620002227)
  otimizado (Gauss-Legendre) : np.float64(69.79093732250087)
  erro relativo               : 1.608e-08
Tempo do calculo vetorizado para os 3 pontos: 50.525s
Erro relativo maximo: 1.608e-08  ->  concordancia de 99.999998%
VALIDACAO OK: resultados concordam com o benchmark original dentro de << 5%.


In [12]:
# def model_function(x, eps, mg, a1, sqrt_s, model_type='log'):

#     m2      = m2_log if model_type == 'log' else m2_pl
#     s_local = sqrt_s ** 2  # [CORREÇÃO 3] s local, correto para 7/8/13 TeV

#     lst_diff_eik = []

#     diff_sigma_eik_opt, amp_eik_opt = diff_sigma_eik_batch(
#     x, mg, a1, eps, m2, sqrt_s, s_local,
#     n_b_points=400, n_q_points_chi=4000,
# )

#     lst_diff_eik.append(diff_sigma_eik_opt)

#     return np.array(lst_diff_eik)


In [13]:
def model_function(x, eps, mg, a1, sqrt_s, model_type='log'):

    m2 = m2_log if model_type == 'log' else m2_pl
    s_local = sqrt_s**2

    diff_sigma_eik_opt, amp_eik_opt = diff_sigma_eik_batch(
        x,
        mg,
        a1,
        eps,
        m2,
        sqrt_s,
        s_local,
        n_b_points=400,
        n_q_points_chi=4000,
    )
    print(f"mg = {mg}, eps = {eps}, a1 = {a1}")
    return diff_sigma_eik_opt

In [14]:
model_function(x_7_atlas, eps_eik, mg_eik, a1_eik, 7000, m2_pl)

mg = 0.93, eps = 0.09583, a1 = 1.4


array([366.43761386, 351.69205118, 333.73824562, 314.28447764,
       294.23939244, 273.84976036, 252.87226412, 232.08723613,
       211.70117804, 192.27022614, 173.85359024, 155.58039208,
       138.30626733, 122.8459537 , 107.93386915,  93.77926502,
        81.3740855 ,  69.79093732])

### Demonstracao de escala: ~5.000 pontos de q2

In [15]:
def model_7(x, eps, mg, a1):
    return model_function(x, eps, mg, a1, sqrt_s=7000, model_type='pl')

def model_8(x, eps, mg, a1):
    return model_function(x, eps, mg, a1, sqrt_s=8000, model_type='pl')

def model_13(x, eps, mg, a1):
    return model_function(x, eps, mg, a1, sqrt_s=13000, model_type='pl')


chi2_7  = LeastSquares(x_7_atlas,  y_7_atlas,  yerr_7_atlas,  model_7, verbose=2)
chi2_8  = LeastSquares(x_8_atlas,  y_8_atlas,  yerr_8_atlas,  model_8, verbose=2)
chi2_13 = LeastSquares(x_13_atlas, y_13_atlas, yerr_13_atlas, model_13, verbose=2)


chi2_total = chi2_7 + chi2_8 + chi2_13	

In [16]:
minuit_eik = Minuit(
    chi2_total,
    mg = mg_eik,
    a1 = a1_eik,
    eps = eps_eik
)


minuit_eik.print_level = 2
minuit_eik.tol = 1e-8
minuit_eik.strategy = 2

minuit_eik.limits['mg'] = (0.3, 0.99)
minuit_eik.limits['a1'] = (0.0001, 5.0)
minuit_eik.limits['eps'] = (0.0001, 0.3)


In [17]:

ncall = 5000

print("Rodando simplex 1...")
minuit_eik.simplex(ncall = ncall)

print("Rodando simplex 2...")
minuit_eik.simplex(ncall = ncall)

print("Rodando simplex 3...")
minuit_eik.simplex(ncall = ncall)



print("Rodando migrad 1...")
minuit_eik.migrad(ncall=ncall)

print("Rodando migrad 2...")
minuit_eik.migrad(ncall=ncall)

print("Rodando migrad 3...")
minuit_eik.migrad(ncall=ncall)

minuit_eik.hesse()


Rodando simplex 1...
mg = 0.93, eps = 0.09583, a1 = 1.4
mg = 0.93, eps = 0.09583, a1 = 1.4
mg = 0.93, eps = 0.09583, a1 = 1.4
(np.float64(0.09583), np.float64(0.93), np.float64(1.4)) -> 30.320799258763387
mg = 0.93, eps = 0.09678957718474368, a1 = 1.4
mg = 0.93, eps = 0.09678957718474368, a1 = 1.4
mg = 0.93, eps = 0.09678957718474368, a1 = 1.4
(np.float64(0.09678957718474368), np.float64(0.93), np.float64(1.4)) -> 38.423849812055224
mg = 0.9389952502169825, eps = 0.09583, a1 = 1.4
mg = 0.9389952502169825, eps = 0.09583, a1 = 1.4
mg = 0.9389952502169825, eps = 0.09583, a1 = 1.4
(np.float64(0.09583), np.float64(0.9389952502169825), np.float64(1.4)) -> 242.80541107695555
mg = 0.93, eps = 0.09583, a1 = 1.4140214571706682
mg = 0.93, eps = 0.09583, a1 = 1.4140214571706682
mg = 0.93, eps = 0.09583, a1 = 1.4140214571706682
(np.float64(0.09583), np.float64(0.93), np.float64(1.4140214571706682)) -> 29.858854508234437
I SimplexBuilder    0 - FCN =       29.85885451 Edm =       212.9465566 NCalls 

┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 17.52 (χ²/ndof = 0.3)      │              Nfcn = 648              │
│ EDM = 6.61e-13 (Goal: 1.19e-10)  │          time = 2930.4 sec           │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ eps  │   0.098   │   0.004   │            │            │ 0.0001  │   0.3   │       │
│ 1 │ mg   │   0.942   │   0.027   │            │            │   0.3   │  0.99   │       │
│ 2 │ a1   │   1.374   │   0.026   │            │            │ 0.0001  │    5    │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌─────┬────────────────────────────┐
│     │      eps       mg       a1 │
├─────┼────────────────────────────┤
│ eps │ 1.88e-05 0.116e-3 0.045e-3 │
│  mg │ 0.116e-3 0.000717   0.3e-3 │
│  a1 │ 0.045e-3   0.3e-3 0.000697 │
└─────┴────────────────────────────┘